# 📱 SMS Spam Detection — Text Mining Pipeline

**Mata Kuliah:** Data Mining / Data Science  
**Dataset:** SMS Spam Collection (UCI ML Repository, ID 228)  
**Tugas:** Melatih satu model machine learning pada dataset publik bersih untuk mendemonstrasikan *(text) data mining*, lalu di-deploy ke aplikasi Streamlit.

---

## Alur notebook
1. **Data Understanding** — memuat pesan, memeriksa distribusi ham/spam & duplikat  
2. **EDA** — panjang pesan, kata paling sering, word cloud  
3. **Data Preparation** — TF-IDF (mengubah teks → vektor angka)  
4. **Modeling** — Logistic Regression (dengan baseline Naive Bayes)  
5. **Evaluation** — precision/recall/F1, confusion matrix, ROC-AUC, cross-validation  
6. **Interpretation** — kata pemicu spam vs ham  
7. **Deployment artifact** — simpan model `.joblib` untuk Streamlit

> Jalankan berurutan: `Runtime ▸ Run all`.

## 0 · Setup

In [ ]:
!pip install -q ucimlrepo wordcloud joblib

import numpy as np, pandas as pd, joblib, json, warnings
import matplotlib.pyplot as plt, seaborn as sns
from wordcloud import WordCloud
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, roc_curve, f1_score)

sns.set_theme(style='whitegrid'); RANDOM_STATE=42
print('Setup selesai ✅')

## 1 · Data Understanding

Memuat data lewat `ucimlrepo`. Bila API bermasalah, sel otomatis beralih mengunduh arsip resmi UCI.

In [ ]:
def load_sms():
    try:
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=228)
        X = ds.data.features; y = ds.data.targets
        df = pd.DataFrame({'message': X.iloc[:,0].astype(str),
                           'label':   y.iloc[:,0].astype(str).str.lower()})
        print('Sumber: ucimlrepo ✅'); return df
    except Exception as e:
        print('ucimlrepo gagal (%s). Beralih ke arsip UCI...' % type(e).__name__)
    import urllib.request, zipfile, io
    url='https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
    raw=urllib.request.urlopen(url, timeout=120).read()
    z=zipfile.ZipFile(io.BytesIO(raw))
    lines=z.read('SMSSpamCollection').decode('utf-8').strip().split('\n')
    rows=[l.split('\t',1) for l in lines]
    print('Sumber: arsip UCI (fallback) ✅')
    return pd.DataFrame(rows, columns=['label','message'])

df = load_sms()
print('Dimensi:', df.shape)
df.head()

In [ ]:
print('Distribusi label:')
print(df['label'].value_counts())
print('\nProporsi spam : %.1f%%' % (100*(df.label=='spam').mean()))
print('Missing value : %d' % df.isna().sum().sum())
print('Pesan duplikat: %d' % df.duplicated().sum())

## 2 · Exploratory Data Analysis

### 2a. Distribusi kelas — dataset timpang
Hanya ~13% pesan berupa spam. Karena itu **akurasi saja menyesatkan** (menebak semua 'ham' sudah 87% benar). Kita akan menilai model dengan **precision, recall, dan F1** pada kelas spam.

In [ ]:
counts=df['label'].value_counts()
plt.figure(figsize=(5,4))
sns.barplot(x=counts.index,y=counts.values,palette=['#4b7a3f','#c0392b'])
plt.title('Distribusi Ham vs Spam'); plt.ylabel('Jumlah pesan')
for i,v in enumerate(counts.values): plt.text(i,v+30,str(v),ha='center')
plt.tight_layout(); plt.show()

### 2b. Panjang pesan — apakah spam lebih panjang?

In [ ]:
df['length']=df['message'].str.len()
plt.figure(figsize=(9,4))
for lbl,col in [('ham','#4b7a3f'),('spam','#c0392b')]:
    sns.kdeplot(df[df.label==lbl]['length'],label=lbl,fill=True,alpha=.4,color=col)
plt.legend(); plt.title('Distribusi Panjang Pesan (karakter)'); plt.xlim(0,300)
plt.tight_layout(); plt.show()
print(df.groupby('label')['length'].mean().round(1))

> **Temuan:** pesan spam cenderung jauh lebih panjang (rata-rata ~139 vs ~71 karakter) — spam berusaha menjejalkan promosi & tautan.

### 2c. Word cloud — kata dominan tiap kelas

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,5))
for a,lbl in zip(ax,['ham','spam']):
    text=' '.join(df[df.label==lbl]['message'])
    wc=WordCloud(width=500,height=350,background_color='white',
                 stopwords={'to','you','the','a','and','is','in','i','u'},colormap='viridis').generate(text)
    a.imshow(wc); a.axis('off'); a.set_title(lbl.upper(),fontsize=14)
plt.tight_layout(); plt.show()

## 3 · Data Preparation — TF-IDF

Model tidak bisa membaca teks mentah, jadi kita ubah tiap pesan menjadi vektor angka dengan **TF-IDF** (*Term Frequency – Inverse Document Frequency*): kata yang sering di satu pesan tapi jarang di seluruh korpus diberi bobot tinggi. Kita pakai unigram+bigram, buang *stopwords* Inggris, dan abaikan kata yang muncul <2 kali.

Split 80/20 **stratified** agar proporsi spam terjaga. TF-IDF dipasang di dalam `Pipeline` supaya hanya belajar kosakata dari data latih (mencegah *data leakage*).

In [ ]:
X = df['message'].astype(str)
y = (df['label']=='spam').astype(int)   # 1 = spam, 0 = ham
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y)
print('Train:',X_train.shape[0],'| Test:',X_test.shape[0])

tfidf = TfidfVectorizer(lowercase=True, stop_words='english',
                        ngram_range=(1,2), min_df=2, sublinear_tf=True)

## 4 · Modeling

Kita bandingkan dua baseline teks klasik, lalu memilih satu sebagai model final.

In [ ]:
def make(clf): return Pipeline([('tfidf',tfidf),('clf',clf)])

candidates={
  'Naive Bayes': MultinomialNB(alpha=0.1),
  'Logistic Regression': LogisticRegression(max_iter=1000,class_weight='balanced',C=10),
}
for name,clf in candidates.items():
    cv=cross_val_score(make(clf),X,y,cv=StratifiedKFold(5,shuffle=True,random_state=RANDOM_STATE),
                       scoring='f1',n_jobs=-1)
    print(f'{name:20s} CV spam-F1 = {cv.mean():.4f} ± {cv.std():.4f}')

> **Logistic Regression** dipilih sebagai model final: F1 sedikit lebih tinggi & stabil, precision/recall seimbang, dan koefisiennya mudah diinterpretasi untuk mengungkap kata pemicu spam.

In [ ]:
model = make(LogisticRegression(max_iter=1000,class_weight='balanced',C=10))
model.fit(X_train,y_train)
print('Model final terlatih ✅')

## 5 · Evaluation

In [ ]:
y_pred=model.predict(X_test)
y_proba=model.predict_proba(X_test)[:,1]
print(classification_report(y_test,y_pred,target_names=['ham','spam'],digits=3))
print('ROC-AUC:',round(roc_auc_score(y_test,y_proba),4))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.5))
cm=confusion_matrix(y_test,y_pred)
ConfusionMatrixDisplay(cm,display_labels=['ham','spam']).plot(ax=ax[0],cmap='Blues',colorbar=False)
ax[0].set_title('Confusion Matrix')
fpr,tpr,_=roc_curve(y_test,y_proba)
ax[1].plot(fpr,tpr,color='#c0392b',lw=2,label='ROC (AUC=%.3f)'%roc_auc_score(y_test,y_proba))
ax[1].plot([0,1],[0,1],'--',color='gray'); ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate'); ax[1].set_title('ROC Curve'); ax[1].legend()
plt.tight_layout(); plt.show()

> **Hasil khas:** spam-F1 ~0,92, akurasi ~0,98, ROC-AUC ~0,99. Confusion matrix menunjukkan sangat sedikit pesan sah yang salah ditandai (*false positive* rendah) — penting agar pesan penting tidak ikut terblokir.

## 6 · Interpretation — kata pemicu

In [ ]:
vec=model.named_steps['tfidf']; clf=model.named_steps['clf']
names=np.array(vec.get_feature_names_out()); coef=clf.coef_[0]
top_spam=names[np.argsort(coef)[-15:]][::-1]; sc_spam=np.sort(coef)[-15:][::-1]
top_ham =names[np.argsort(coef)[:15]];       sc_ham =np.sort(coef)[:15]

fig,ax=plt.subplots(1,2,figsize=(13,5))
ax[0].barh(top_spam[::-1],sc_spam[::-1],color='#c0392b'); ax[0].set_title('Kata paling menandai SPAM')
ax[1].barh(top_ham[::-1],np.abs(sc_ham[::-1]),color='#4b7a3f'); ax[1].set_title('Kata paling menandai HAM')
plt.tight_layout(); plt.show()
print('Top spam:',list(top_spam[:10]))

> **Insight:** spam bertumpu pada ajakan bertransaksi & nomor pendek — *txt, claim, won, 150p, mobile, www, reply, stop*. Ham berisi kosakata percakapan sehari-hari — *ok, home, hey, later, sorry*. Model belajar pola yang sangat intuitif.

## 7 · Menyimpan model untuk Streamlit

In [ ]:
metrics={'spam_f1':float(f1_score(y_test,y_pred)),
         'accuracy':float((y_pred==y_test).mean()),
         'roc_auc':float(roc_auc_score(y_test,y_proba))}
joblib.dump({'model':model,'metrics':metrics}, 'sms_spam_model.joblib', compress=3)
print('Tersimpan: sms_spam_model.joblib', metrics)

try:
    from google.colab import files; files.download('sms_spam_model.joblib')
except Exception: print('Bukan Colab — file ada di direktori kerja.')

---
## Ringkasan

| Tahap | Hasil |
|---|---|
| Dataset | 5.572 SMS · 13,4% spam · 0 missing |
| Representasi | TF-IDF (unigram+bigram) |
| Model | Logistic Regression (class-weight balanced) |
| Spam-F1 | ~0,92 |
| Akurasi | ~0,98 |
| ROC-AUC | ~0,99 |

Langkah berikutnya: jalankan `app.py` dengan `sms_spam_model.joblib` dari sel di atas.